In [ ]:
# 1. Project/config — chỉ điều phối, business logic nằm trong src/data_pipeline.py
from pathlib import Path
import os, sys
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(path for path in candidates if (path / 'config' / 'config.yaml').is_file())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
from src.data_pipeline import DataPreparationPipeline
pipeline = DataPreparationPipeline(PROJECT_ROOT)
pipeline.config

In [ ]:
# 2. Load raw data
source_info = pipeline.load_raw_data()
source_info

In [ ]:
# 3. Audit source data
source_audit = pipeline.audit_source_data()
source_audit

In [ ]:
# 4. Build daily disease counts
daily_cases = pipeline.build_daily_disease_counts()
daily_cases.head()

In [ ]:
# 5. Weather current + exact trailing 3d/7d
weather_features = pipeline.build_weather_current_3d_7d()
weather_features.dropna().head()

In [ ]:
# 6. Context/query dataset
contexts = pipeline.build_query_contexts()
contexts.head()

In [ ]:
# 7. Sparse targets for H3/H7/H14
targets = pipeline.build_horizon_targets()
targets.head()

In [ ]:
# 8. Chronological split with H14 purge
split_frames = pipeline.build_temporal_splits()
pipeline.split_metadata

In [ ]:
# 9. Train-only disease support
disease_catalog = pipeline.calculate_train_support()
pipeline.support_counts

In [ ]:
# 10. Validation/assertions
checks = pipeline.validate_all()
assert all(checks.values())
checks

In [ ]:
# 11. Three real-date examples
examples = pipeline.build_examples()
examples

In [ ]:
# 12. Save artifacts and preparation report — no model training
metadata = pipeline.save_artifacts()
assert metadata['no_model_trained'] is True
{'queries': metadata['contexts']['queries'], 'report': str(PROJECT_ROOT / 'reports' / 'DATA_PREPARATION_REPORT.md'), 'model_trained': False}